# 16. Ders: İstem Mühendisliği ve Yılın Şiiri Projesi (Bölüm 2)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Bir restorana gidip garsona sadece "Bana yemek getir" derseniz, garson size muhtemelen istemediğiniz bir şey getirecektir veya şaşkınlıkla size bakacaktır. Ancak "Az pişmiş, yanında fırın patates olan, tuzsuz bir antrikot istiyorum" derseniz tam olarak istediğinizi alırsınız. 

Yapay zeka modelleri de tıpkı böyledir. Önceki derste GPT-2'ye sadece bir cümle verdik ve kafasına göre (bazen anlamsız) şeyler yazdı. Modeli istediğimiz sınırlar içinde tutmak, ona bir 'rol' vermek ve kurallar koymak tamamen bir mühendislik işidir. Buna 'İstem Mühendisliği' (Prompt Engineering) denir. Bugün Yılın Şiiri projemizi bu mühendislik ilkeleriyle tamamlayacağız.

## Adım 2: Teknik Terimler ve Sözlük

- Prompt Engineering (İstem Mühendisliği): Yapay zekadan en doğru, güvenli ve kaliteli çıktıyı almak için girdi metinlerini (prompt) stratejik olarak tasarlama bilimidir.
- Context (Bağlam): Modelin konuyu ve rolünü anladığı arka plan hikayesidir.
- Zero-Shot Learning: Modele hiçbir örnek vermeden doğrudan görevi yapmasını istemektir (Hata payı yüksektir).
- Few-Shot Learning: Modele görevi vermeden önce, içine birkaç tane 'doğru soru-cevap örneği' ekleyerek şablonu (pattern) kopyalamasını sağlama tekniğidir (Başarı oranı çok yüksektir).

## Adım 3: GPT-2 Modelimizi Tekrar Hazırlayalım

Bu dersimizde de kelimeler üreteceğimiz için `text-generation` aracımızı (GPT-2) kullanacağız.

In [ ]:
from transformers import pipeline
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

yazar_model = pipeline("text-generation", model="gpt2")
print("Yapay Zeka Yazar (GPT-2) hazır.")

## Adım 4: Kötü Bir Prompt İle İyi Bir Prompt Arasındaki Fark

Modele sadece "Yapay zeka hakkında şiir yaz" (Write a poem about AI) dediğimizde (Zero-Shot) genellikle ne yapacağını bilemez veya düz metin (makale gibi) yazar. Ona bir **Rol** vermeliyiz.

In [ ]:
# KÖTÜ PROMPT (Zero-Shot, Rolsüz)
kotu_prompt = "Write a poem about artificial intelligence:"

# İYİ PROMPT (Rol-Prompting ve Sınır Çizme)
iyi_prompt = """
You are William Shakespeare. Write a beautiful, short and emotional poem about artificial intelligence.
Poem:
"""

# İkisini de test edip sonuçları kıyaslayalım (Temperature'ı 0.7'de tutuyoruz)
sonuc_kotu = yazar_model(kotu_prompt, max_new_tokens=40, temperature=0.7, pad_token_id=50256)
sonuc_iyi = yazar_model(iyi_prompt, max_new_tokens=40, temperature=0.7, pad_token_id=50256)

print("--- KÖTÜ PROMPT ÇIKTISI ---")
print(sonuc_kotu[0]['generated_text'])
print("\n" + "*"*40 + "\n")
print("--- İYİ PROMPT ÇIKTISI ---")
print(sonuc_iyi[0]['generated_text'])

## Adım 5: Few-Shot Learning (Birkaç Örnekle Öğrenme) Tekniği

Eğer modelin belirli bir şablonda (örneğin kelime-anlam eşleştirmesi) cevap vermesini istiyorsak, promptun içine önce bizim yazdığımız 2-3 örneği koyarız. Model bu örnekleri (Few-Shot) görür ve son satırdaki sorunun cevabını aynı şablonda üretir.

In [ ]:
# Few-Shot Prompt Tasarımı
few_shot_prompt = """
Convert words into emotions.
Word: Sun -> Emotion: Happy
Word: Rain -> Emotion: Sad
Word: War -> Emotion: Angry
Word: Robot -> Emotion: """

# Burada modelin fazla kelime üretmesini istemiyoruz, 
# sadece son kelimeyi (Örn: Smart, Cold, Curious) tahmin etmesini istiyoruz.
sonuc_few_shot = yazar_model(few_shot_prompt, max_new_tokens=3, temperature=0.3, pad_token_id=50256)

print("--- FEW-SHOT ÖĞRENME ÇIKTISI ---")
print(sonuc_few_shot[0]['generated_text'])

## Adım 6: Yılın Şiiri Projesini (Gelişmiş Prompt ile) Kurma

Şimdi kullanıcının verdiği 3 kelimeyi alıp, bunu sağlam bir İngilizce 'Yaratıcı Yazarlık' promptuna dönüştüren tam otomatik fonksiyonumuzu (Yılın Şiiri motoru) yazalım.

In [ ]:
def yilin_siiri_yazdir(kelime1, kelime2, kelime3):
    print(f"[{kelime1}, {kelime2}, {kelime3}] kelimeleriyle ilham aranıyor...")
    
    # Rol ve Bağlam içeren gelişmiş Prompt tasarımı
    siir_promptu = f"""
You are an award-winning poet.
Write a short, beautiful, and inspiring poem using the themes of {kelime1}, {kelime2}, and {kelime3}.

Poem:
"""
    
    # Modelin üretime geçmesi
    siir_ciktisi = yazar_model(siir_promptu, max_new_tokens=50, temperature=0.8, pad_token_id=50256)
    ham_metin = siir_ciktisi[0]['generated_text']
    
    # Adım 7: Çıktıyı Temizleme (Prompt kısmını kesip atma)
    # Model genellikle promptu ve cevabı birleşik döndürür. '.split("Poem:")' diyerek 
    # yazılan şiiri oradan koparıp alıyoruz.
    temiz_siir = ham_metin.split("Poem:")[1].strip()
    
    print("\n🏆 YILIN ŞİİRİ 🏆")
    print(temiz_siir)
    print("-----------------")

# Test edelim (Kelimeleri İngilizce vermeliyiz çünkü model GPT-2)
yilin_siiri_yazdir("ocean", "stars", "dreams")

## Adım 8: Dene ve Keşfet - Farklı Roller

Adım 6'daki `siir_promptu` değişkeninin içindeki "You are an award-winning poet" kısmını silin ve yerine "You are an angry pirate" (Sen kızgın bir korsansın) veya "You are an alien" (Sen bir uzaylısın) yazarak kodu tekrar çalıştırın. Şiirin tarzının ne kadar inanılmaz bir şekilde değiştiğini gözlemleyin.

## Adım 9: Sıra Sende Görevi - Matematik Öğretmeni Botu

Problem: GPT-2'nin kullanıcının sorduğu basit matematik sorularına (İngilizce) bir öğretmen gibi kibarca cevap vermesini sağlayacak bir **Few-Shot Prompt** yazmanız isteniyor.

Gereksinimler:
1. `matematik_sor(soru)` adında bir fonksiyon oluşturun.
2. Fonksiyonun içinde `ogretmen_promptu` hazırlayın.
3. Promptun içine şu kurguyu ekleyin (Few-Shot):
   `You are a polite math teacher.`
   `Student: What is 2 + 2? -> Teacher: The answer is 4, my dear student.`
   `Student: What is 5 + 5? -> Teacher: The answer is 10, my dear student.`
   `Student: {soru} -> Teacher:`
4. Modeli `max_new_tokens=10` ve `temperature=0.2` (Matematikte çılgınlık istenmez) ile çalıştırın.
5. Çıktıyı `split("Teacher:")` kullanarak temizleyip (sadece son parçayı) ekrana yazdırın.
6. `matematik_sor("What is 10 + 10?")` diyerek test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 16. Dersi tamamlayarak günümüzün en değerli teknoloji becerilerinden biri olan 'Prompt Engineering' alanına giriş yaptınız. Modelleri sadece kullanmayı değil, onları terbiye etmeyi öğrendiniz.

Bir sonraki dersimizde yapay zekanın karanlık yüzüne (toksisite, ırkçılık, veri kirliliği) bakacağız ve 'Watchdog' (Veri Bekçisi) projesine başlayacağız.